### Endpoint for SAFE conversion process in DPR service that takes as input a path to a SAFE product in a bucket, calls CPM to transform the product, then returns the path to the converted product.

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-675


In [ ]:
# Init environment before running a demo notebook.
import os
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

In [ ]:
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if cluster_mode:
    os.environ["DASK_GATEWAY_EOPF_ADDRESS"] = os.environ["DASK_GATEWAY_ADDRESS"]
os.environ["DASK_CLUSTER_EOPF_NAME"] = dask_cluster.name

from rs_client.rs_client import RsClient
rs_server_href = os.getenv("RSPY_WEBSITE")
rs_server_api_key = os.environ.get("RSPY_APIKEY")
generic_client = RsClient(rs_server_href, rs_server_api_key, OWNER_ID, None)

In [ ]:
legacy_products = [
    "S3A_OL_1_EFR____20240626T125108_20240626T125215_20240626T141905_0067_114_052_3780_PS1_O_NR_004.SEN3",
    "S3A_OL_2_LFR____20240430T083943_20240430T084243_20240501T092030_0179_112_007_2160_PS1_O_NT_002.SEN3.zip",
    "S3B_OL_2_LFR____20240430T130043_20240430T130343_20240501T004249_0179_092_252_1980_PS2_O_NT_002.SEN3",
    "S3B_SY_2_V10____20240611T000000_20240620T235959_20240622T121945_EUROPE____________PS2_O_ST_002.SEN3",
    "S1A_IW_SLC__1SDV_20221231T100709_20221231T100736_046574_0594DE_0A58.SAFE.zip",
]

payloads = [
    {
        "input_safe_path": f"s3://rs-dev-cluster-temp/conversion/{legacy_product}",
        "output_zarr_dir_path": "s3://rs-dev-cluster-temp/conversion/",
    }
    for legacy_product in legacy_products
]

In [ ]:
dpr_client = generic_client.get_dpr_client()

# Launch all conversion jobs - "/dpr/processes/{resource}/execution" endpoint with "conv_safe_zarr" as resource
job_status_list = [dpr_client.run_conv_safe_zarr(payload) for payload in payloads]

results = [dpr_client.wait_for_job(job_status, logger=None, job_name="Conversion Processor") for job_status in job_status_list]
results

In [ ]:
responses = [requests.get(f"{dpr_client.href_service}/dpr/jobs/{job_status['jobID']}").json() for job_status in job_status_list]
responses

In [ ]:
shutdown = False
if shutdown:
    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

    # Close the python objects
    close_dask_clusters()